# Stage 2: Industrial Textile Wastewater RO Baseline
**Project:** AI-Enabled Digital Twin for Fouling-Aware Optimization of Textile Wastewater Reuse  
**Reference Study:** Sowgath, Sarker & Mujtaba (2025), *"Design of Multi Stage Reverse Osmosis Process for Reuse of Textile Wastewater"*, Chemical Engineering Transactions, Vol. 117.  
**Industrial Case:** Nice Cotton Ltd., Bangladesh (MBR–RO Effluent Reclamation)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))

from ro_model.membrane import MembraneElementProperties, SimulationConfig
from ro_model.vessel import PressureVessel
from ro_model.stage import ROStage
from ro_model.system import ROSystem
from ro_model.water_quality import WaterQualityStream, ApparentRejectionProfile


## 1. Published Industrial Stream Data & Consistency Audit
Loading Nice Cotton Ltd. stream data and auditing mathematical consistency against global mass conservation.


In [ ]:
streams_df = pd.read_csv('../data/raw/nice_cotton_streams.csv')
print("=== Published Stream Data (Table 1, Sowgath et al., 2025) ===")
display(streams_df)

# Audit calculations
cf_tds = 2041.0
cp_tds = 18.0
cr_tds = 3064.0

sr_calc = (1.0 - cp_tds / cf_tds) * 100.0
cr_theor_70 = (cf_tds - 0.70 * cp_tds) / 0.30
wr_implied = (cr_tds - cf_tds) / (cr_tds - cp_tds) * 100.0

print(f"\n1. Apparent Salt Rejection from reported stream TDS: {sr_calc:.2f}% (vs paper narrative benchmark ~73%)")
print(f"2. Theoretical Reject TDS required at 70% recovery: {cr_theor_70:.2f} mg/L (vs reported {cr_tds:.2f} mg/L, 54.7% deficit)")
print(f"3. Implied recovery if Cr = 3064 mg/L were true: {wr_implied:.2f}% (vs target 70.0%)")


## 2. Multi-Stage Industrial Plant Simulation
Simulating the 2-stage RO process with 3 elements per pressure vessel in series.


In [ ]:
# Physical Transport Properties (Toray TML20D-400)
props_mfr = MembraneElementProperties(
    membrane_area_m2=37.0,
    Aw_m_pa_s=1.0232e-11,  # 3.6835 LMH/bar (Manufacturer reconciled)
    As_m_s=1.7827e-8       # 0.06418 LMH (Manufacturer nominal rejection)
)

feed_stream = WaterQualityStream(
    flow_m3_s=30.0 / 3600.0,
    tds_mg_l=2041.0,
    cod_mg_l=51.0,
    bod_mg_l=7.0,
    tss_mg_l=3.0,
    colour_pt_co=300.0,
    ph=8.0
)

# 2-Stage System Configuration (3:2 Array = 15 elements)
s1 = ROStage(name="Stage_1", parallel_vessels=3, elements_per_vessel=3, element_properties=props_mfr)
s2 = ROStage(name="Stage_2", parallel_vessels=2, elements_per_vessel=3, element_properties=props_mfr)
system = ROSystem(name="Nice_Cotton_2Stage_RO", stages=[s1, s2], topology="concentrate_to_stage2")

# Solve under P1 = 13.0 bar, P2 = 18.0 bar (+5.0 bar interstage boost)
res = system.solve(
    feed_flow_m3_hr=30.0,
    feed_tds_mg_l=2041.0,
    stage_pressures_bar=[13.0, 18.0],
    feed_quality_stream=feed_stream
)

print(f"=== 2-STAGE INDUSTRIAL PLANT RESULTS ===")
print(f"Overall Water Recovery  : {res.overall_water_recovery_percent:.2f}% (Target: 70.0%)")
print(f"Permeate Flow (Qp)      : {res.permeate_flow_m3_hr:.2f} m3/h")
print(f"Concentrate Flow (Qr)   : {res.concentrate_flow_m3_hr:.2f} m3/h")
print(f"Permeate TDS (Cp)       : {res.permeate_tds_mg_l:.2f} mg/L (Published: 18.0 mg/L)")
print(f"Concentrate TDS (Cr)    : {res.concentrate_tds_mg_l:.2f} mg/L (Theoretical mass balance: 6761.33 mg/L)")
print(f"Overall Salt Rejection  : {res.overall_salt_rejection_percent:.4f}%")
print(f"Average System Flux     : {res.average_system_flux_lmh:.2f} LMH")
print(f"System SEC              : {res.system_sec_kwh_per_m3:.4f} kWh/m3")
print(f"Water Balance Residual  : {res.water_mass_balance_error_m3_s:.2e} m3/s")
print(f"Solute Balance Residual : {res.solute_mass_balance_error_kg_s:.2e} kg/s")


## 3. Comparison Table
Comparing Published Industrial Data vs Uncalibrated Physical Model vs 70% Recovery Feasible Design.


In [ ]:
comp_df = pd.read_csv('../results/stage2/tables/textile_comparison.csv')
display(comp_df)
